In [ ]:
# =============================================
# DAY 36 — HYPERPARAMETER TUNING (FINAL VERSION)
# =============================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("=== DAY 36 — HYPERPARAMETER TUNING ===\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================
# 1. LOAD DATA
# =============================================
state_data = pd.read_csv("../data/state_dataset.csv", index_col=0)
market_data = pd.read_csv("../data/market_master.csv", index_col=0)

state_data.index = pd.to_datetime(state_data.index)
market_data.index = pd.to_datetime(market_data.index)

data = state_data.join(market_data[["nifty_ret"]], how="inner")

train_data = data.loc["2019":"2022"]

print(f"Train Data Shape: {train_data.shape}")

# =============================================
# 2. ENVIRONMENT
# =============================================
class QuantumAlphaEnv:
    def __init__(self, data):
        self.data = data.reset_index(drop=True)
        self.n_features = data.shape[1] - 1
        self.max_steps = len(data) - 1
        self.reset()
    
    def reset(self):
        self.current_step = 0
        self.position = 0
        self.balance = 1.0
        self.peak = 1.0
        return self._get_observation()
    
    def _get_observation(self):
        obs = self.data.iloc[self.current_step].values[:-1]
        return np.append(obs, self.position)
    
    def step(self, action):
        prev_position = self.position
        self.position = action - 1
        
        ret = self.data.iloc[self.current_step]["nifty_ret"]
        cost = abs(self.position - prev_position) * 0.0005
        net_ret = self.position * ret - cost
        
        self.balance *= (1 + net_ret)
        self.peak = max(self.peak, self.balance)
        
        drawdown = (self.balance - self.peak) / self.peak
        
        reward = net_ret - 0.08*(net_ret**2)
        if drawdown < -0.05:
            reward -= 0.25 * abs(drawdown)
        
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self._get_observation(), reward, done, {"net_ret": net_ret}

# =============================================
# 3. DQN MODEL
# =============================================
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
    
    def forward(self, x):
        return self.net(x)

# =============================================
# 4. REPLAY BUFFER
# =============================================
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, *args):
        self.buffer.append(args)
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(actions).to(device),
            torch.FloatTensor(rewards).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(dones).to(device)
        )
    
    def __len__(self):
        return len(self.buffer)

# =============================================
# 5. TRAIN FUNCTION
# =============================================
def train_agent(env, gamma, lr, epsilon_decay, batch_size, episodes=80):
    
    state_size = env.n_features + 1
    action_size = 3
    
    policy_net = DQN(state_size, action_size).to(device)
    target_net = DQN(state_size, action_size).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    criterion = nn.SmoothL1Loss()
    
    memory = ReplayBuffer()
    
    epsilon = 1.0
    epsilon_min = 0.05
    
    warmup = 300
    target_update = 300
    
    global_step = 0
    rewards_history = []
    
    for ep in range(episodes):
        
        state = env.reset()
        total_reward = 0
        
        while True:
            
            if random.random() < epsilon:
                action = random.randint(0, 2)
            else:
                with torch.no_grad():
                    s = torch.FloatTensor(state).unsqueeze(0).to(device)
                    action = policy_net(s).argmax().item()
            
            next_state, reward, done, _ = env.step(action)
            
            memory.push(state, action, reward, next_state, done)
            state = next_state
            
            total_reward += reward
            global_step += 1
            
            if len(memory) > warmup:
                states, actions, rewards, next_states, dones = memory.sample(batch_size)
                
                current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
                
                with torch.no_grad():
                    next_q = target_net(next_states).max(1)[0]
                    target_q = rewards + gamma * next_q * (1 - dones)
                
                loss = criterion(current_q, target_q)
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()
            
            if global_step % target_update == 0:
                target_net.load_state_dict(policy_net.state_dict())
            
            if done:
                break
        
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_reward)
    
    score = np.mean(rewards_history[-10:])
    
    return score, policy_net

# =============================================
# 6. HYPERPARAMETER CONFIGS
# =============================================
configs = [
    {"gamma":0.99, "lr":1e-4, "decay":0.995, "bs":128},
    {"gamma":0.95, "lr":1e-4, "decay":0.995, "bs":128},
    {"gamma":0.99, "lr":5e-4, "decay":0.995, "bs":128},
    {"gamma":0.99, "lr":1e-4, "decay":0.999, "bs":128},
]

# =============================================
# 7. RUN TUNING
# =============================================
train_env = QuantumAlphaEnv(train_data)

results = []
best_score = -np.inf
best_model = None
best_config = None

print("\n=== START TUNING ===\n")

for cfg in configs:
    
    print(f"Testing: {cfg}")
    
    score, model = train_agent(
        train_env,
        gamma=cfg["gamma"],
        lr=cfg["lr"],
        epsilon_decay=cfg["decay"],
        batch_size=cfg["bs"]
    )
    
    print(f"Score: {score:.2f}\n")
    
    results.append({**cfg, "score": score})
    
    if score > best_score:
        best_score = score
        best_model = model
        best_config = cfg

# =============================================
# 8. RESULTS
# =============================================
results_df = pd.DataFrame(results).sort_values("score", ascending=False)

print("\n=== RESULTS ===")
print(results_df)

print("\nBest Config:")
print(best_config)

# =============================================
# 9. FINAL TRAINING
# =============================================
print("\n=== FINAL TRAINING ===\n")

final_env = QuantumAlphaEnv(train_data)

_, final_model = train_agent(
    final_env,
    gamma=best_config["gamma"],
    lr=best_config["lr"],
    epsilon_decay=best_config["decay"],
    batch_size=best_config["bs"],
    episodes=200
)

# =============================================
# 10. SAVE MODEL
# =============================================
torch.save(final_model.state_dict(), "../models/dqn_final_tuned.pth")

print("\n FINAL MODEL SAVED")

# =============================================
# 11. VISUALIZATION
# =============================================
plt.figure(figsize=(8,5))
plt.bar(range(len(results_df)), results_df["score"])
plt.title("Hyperparameter Comparison")
plt.show()

=== DAY 36 — HYPERPARAMETER TUNING ===

Train Data Shape: (736, 7)

=== START TUNING ===

Testing: {'gamma': 0.99, 'lr': 0.0001, 'decay': 0.995, 'bs': 128}
